In [ ]:
# Example 1: Basic ArgumentParser using RichHelpFormatter.
import argparse
from rich_argparse import RichHelpFormatter

parser = argparse.ArgumentParser(
    prog='demo',
    description='A demo CLI using rich_argparse.',
    formatter_class=RichHelpFormatter,
)
parser.add_argument('--name', help='your name', default='world')
parser.add_argument('--count', type=int, default=1, help='how many times to greet')
parser.add_argument('-v', '--verbose', action='store_true', help='increase verbosity')

help_text = parser.format_help()
print(help_text)
assert 'demo' in help_text
assert '--name' in help_text


In [ ]:
# Example 2: Alternate formatters - Raw / RawDescription / ArgumentDefaults / MetavarType.
import argparse
from rich_argparse import (
    RawDescriptionRichHelpFormatter,
    RawTextRichHelpFormatter,
    ArgumentDefaultsRichHelpFormatter,
    MetavarTypeRichHelpFormatter,
)

formatters = [
    RawDescriptionRichHelpFormatter,
    RawTextRichHelpFormatter,
    ArgumentDefaultsRichHelpFormatter,
    MetavarTypeRichHelpFormatter,
]
for fmt in formatters:
    parser = argparse.ArgumentParser(prog='p', formatter_class=fmt)
    parser.add_argument('--n', type=int, default=3, help='count')
    out = parser.format_help()
    assert '--n' in out, f'missing flag for {fmt.__name__}'
    print(f'{fmt.__name__}: OK')


In [ ]:
# Example 3: Customize style theme via RichHelpFormatter.styles.
import argparse
from rich_argparse import RichHelpFormatter

# Snapshot, mutate, restore so other cells aren't affected.
original = dict(RichHelpFormatter.styles)
try:
    RichHelpFormatter.styles['argparse.args'] = 'bold cyan'
    RichHelpFormatter.styles['argparse.groups'] = 'bold magenta'
    RichHelpFormatter.styles['argparse.help'] = 'italic green'

    parser = argparse.ArgumentParser(prog='styled', formatter_class=RichHelpFormatter)
    parser.add_argument('--foo', help='styled help text')
    out = parser.format_help()
    assert '--foo' in out
    print(out)
finally:
    RichHelpFormatter.styles = original


In [ ]:
# Example 4: Rich markup in description / epilog via class flags.
import argparse
from rich_argparse import RichHelpFormatter

RichHelpFormatter.usage_markup = True
try:
    parser = argparse.ArgumentParser(
        prog='markup-demo',
        description='[bold]Bold description[/] and [italic]italic[/] text.',
        epilog='[red]See docs for details.[/]',
        formatter_class=RichHelpFormatter,
    )
    parser.add_argument('--mode', choices=['fast', 'slow'], help='operating mode')
    out = parser.format_help()
    assert 'description' in out.lower() or 'Bold description' in out
    assert '--mode' in out
    print(out)
finally:
    RichHelpFormatter.usage_markup = False


In [ ]:
# Example 5: Subparsers + argument groups + parse_args round-trip.
import argparse
from rich_argparse import RichHelpFormatter

parser = argparse.ArgumentParser(prog='tool', formatter_class=RichHelpFormatter)
group = parser.add_argument_group('global options')
group.add_argument('--config', help='path to config file')

sub = parser.add_subparsers(dest='cmd', required=True)
p_run = sub.add_parser('run', help='run something', formatter_class=RichHelpFormatter)
p_run.add_argument('target', help='what to run')
p_list = sub.add_parser('list', help='list things', formatter_class=RichHelpFormatter)
p_list.add_argument('--all', action='store_true')

ns = parser.parse_args(['--config', 'cfg.yml', 'run', 'job-1'])
assert ns.cmd == 'run'
assert ns.target == 'job-1'
assert ns.config == 'cfg.yml'

help_out = parser.format_help()
# rich-argparse title-cases group headings ("Global Options:").
assert 'Global Options' in help_out
assert '{run,list}' in help_out or 'run' in help_out
print('parse_args + subparser help: OK')
